> Menyiapkan Dataset

In [1]:
import numpy as np
import pandas as pd

np.random.seed(111)
n = 2500
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas11 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "harga_satuan": np.random.choice([15000, 18000, 20000, 22000, 25000, 30000], size=n),
})
data_tugas11["total_penjualan"] = data_tugas11["qty"] * data_tugas11["harga_satuan"] + np.random.normal(0, 8000, size=n)
data_tugas11.to_csv("tugas11_penjualan.csv", index=False)

print(f"Dataset berhasil dibuat: {len(data_tugas11)} baris")
print(data_tugas11.head())

Dataset berhasil dibuat: 2500 baris
        nama_menu      cabang waktu_pesan  qty  harga_satuan  total_penjualan
0  Kentang Goreng    Semarang       Malam    1         30000     27750.163532
1  Kentang Goreng  Yogyakarta        Sore    3         20000     64216.160744
2  Kentang Goreng    Semarang       Siang    5         22000    108722.792592
3  Kentang Goreng  Yogyakarta        Pagi    2         20000     54483.205872
4      Roti Bakar    Magelang       Malam    2         30000     58157.944120


Menyiapkan SparkSession

In [2]:
from pyspark.sql import SparkSession

from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

spark = SparkSession.builder \
    .appName("BIGDATA 11 - Tugas Mandiri") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

df_tugas = spark.read.csv(
    "tugas11_penjualan.csv",
    header=True,
    inferSchema=True
)

df_tugas.show(5)
print("Total data:", df_tugas.count())

26/11/19 22:32:20 WARN Utils: Your hostname, pathann resolves to a loopback address: 127.0.1.1; using 192.168.0.121 instead (on interface wlp8s0)
26/11/19 22:32:20 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/19 22:32:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/19 22:32:28 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


+--------------+----------+-----------+---+------------+------------------+
|     nama_menu|    cabang|waktu_pesan|qty|harga_satuan|   total_penjualan|
+--------------+----------+-----------+---+------------+------------------+
|Kentang Goreng|  Semarang|      Malam|  1|       30000|27750.163531558544|
|Kentang Goreng|Yogyakarta|       Sore|  3|       20000| 64216.16074371864|
|Kentang Goreng|  Semarang|      Siang|  5|       22000|108722.79259154423|
|Kentang Goreng|Yogyakarta|       Pagi|  2|       20000|54483.205871792496|
|    Roti Bakar|  Magelang|      Malam|  2|       30000| 58157.94412015849|
+--------------+----------+-----------+---+------------+------------------+
only showing top 5 rows

Total data: 2500


> 1 A: Feature Engineering

In [3]:
# Mengubah nama menu menjadi angka
indexer_menu = StringIndexer(
    inputCol="nama_menu",
    outputCol="nama_menu_idx"
)

# Mengubah cabang menjadi angka
indexer_cabang = StringIndexer(
    inputCol="cabang",
    outputCol="cabang_idx"
)

# Mengubah waktu pemesanan menjadi angka
indexer_waktu = StringIndexer(
    inputCol="waktu_pesan",
    outputCol="waktu_pesan_idx"
)

# Mempelajari kategori dan menerapkannya ke dataset
model_indexer_menu = indexer_menu.fit(df_tugas)
model_indexer_cabang = indexer_cabang.fit(df_tugas)
model_indexer_waktu = indexer_waktu.fit(df_tugas)

df_indexed = model_indexer_menu.transform(df_tugas)
df_indexed = model_indexer_cabang.transform(df_indexed)
df_indexed = model_indexer_waktu.transform(df_indexed)

# Menggabungkan seluruh fitur
assembler = VectorAssembler(
    inputCols=[
        "qty",
        "harga_satuan",
        "nama_menu_idx",
        "cabang_idx",
        "waktu_pesan_idx"
    ],
    outputCol="features"
)

df_final = assembler.transform(df_indexed)

df_final.select(
    "features",
    "total_penjualan"
).show(5, truncate=False)

+-------------------------+------------------+
|features                 |total_penjualan   |
+-------------------------+------------------+
|[1.0,30000.0,5.0,2.0,1.0]|27750.163531558544|
|[3.0,20000.0,5.0,1.0,3.0]|64216.16074371864 |
|[5.0,22000.0,5.0,2.0,2.0]|108722.79259154423|
|[2.0,20000.0,5.0,1.0,0.0]|54483.205871792496|
|[2.0,30000.0,4.0,0.0,1.0]|58157.94412015849 |
+-------------------------+------------------+
only showing top 5 rows



> B: Train/Test Split dan Pelatihan Model

In [4]:
# Membagi data menjadi 75% training dan 25% testing
train, test = df_final.randomSplit(
    [0.75, 0.25],
    seed=101
)

print("Data Training:", train.count())
print("Data Testing :", test.count())

# Membuat model Regresi Linear
lr = LinearRegression(
    featuresCol="features",
    labelCol="total_penjualan"
)

# Melatih model
model = lr.fit(train)

# Membuat prediksi
predictions = model.transform(test)

predictions.select(
    "qty",
    "harga_satuan",
    "total_penjualan",
    "prediction"
).show(10)

Data Training: 1839
Data Testing : 661
+---+------------+------------------+------------------+
|qty|harga_satuan|   total_penjualan|        prediction|
+---+------------+------------------+------------------+
|  1|       25000|19721.170446051627|31056.146105506836|
|  1|       25000|  21236.0956379149|31056.146105506836|
|  4|       22000|  85834.8564401191| 88127.37331410211|
|  4|       22000|102174.09100317833| 88127.37331410211|
|  4|       30000|113044.63431555482|112310.74025938749|
|  5|       22000|115810.29600214415|110174.03658512786|
|  5|       25000|140386.40273615732|119242.79918960987|
|  2|       30000| 73842.64420790318| 67877.77492201277|
|  3|       15000|32345.642983082646| 44580.62517062847|
|  4|       15000| 45510.16712269087| 66627.28844165422|
+---+------------+------------------+------------------+
only showing top 10 rows



> C: Evaluasi Model

In [5]:
# Menghitung RMSE
evaluator_rmse = RegressionEvaluator(
    labelCol="total_penjualan",
    predictionCol="prediction",
    metricName="rmse"
)

rmse = evaluator_rmse.evaluate(predictions)

# Menghitung R²
evaluator_r2 = RegressionEvaluator(
    labelCol="total_penjualan",
    predictionCol="prediction",
    metricName="r2"
)

r2 = evaluator_r2.evaluate(predictions)

print(f"RMSE : Rp {rmse:,.0f}")
print(f"R²   : {r2:.4f}")

RMSE : Rp 10,901
R²   : 0.9107


> D: Eksperimen Perbandingan Fitur

In [6]:
# Membuat VectorAssembler dengan dua fitur
assembler_sederhana = VectorAssembler(
    inputCols=[
        "qty",
        "harga_satuan"
    ],
    outputCol="features_sederhana"
)

df_sederhana = assembler_sederhana.transform(df_indexed)

# Membagi data training dan testing
train_sederhana, test_sederhana = df_sederhana.randomSplit(
    [0.75, 0.25],
    seed=101
)

# Membuat model kedua
lr_sederhana = LinearRegression(
    featuresCol="features_sederhana",
    labelCol="total_penjualan"
)

# Melatih model kedua
model_sederhana = lr_sederhana.fit(train_sederhana)

# Membuat prediksi
prediksi_sederhana = model_sederhana.transform(test_sederhana)

# Menghitung R² model kedua
evaluator_r2_sederhana = RegressionEvaluator(
    labelCol="total_penjualan",
    predictionCol="prediction",
    metricName="r2"
)

r2_sederhana = evaluator_r2_sederhana.evaluate(
    prediksi_sederhana
)

print("R² Model Utama      :", r2)
print("R² Model Sederhana  :", r2_sederhana)

R² Model Utama      : 0.910679109181288
R² Model Sederhana  : 0.9107572632054388


> E: Prediksi Skenario Baru

In [7]:
# Membuat tiga data pesanan baru
data_baru = [
    ("Kopi Susu", "Magelang", "Pagi", 2, 15000),
    ("Americano", "Yogyakarta", "Siang", 3, 20000),
    ("Cappuccino", "Semarang", "Malam", 1, 25000)
]

df_baru = spark.createDataFrame(
    data_baru,
    [
        "nama_menu",
        "cabang",
        "waktu_pesan",
        "qty",
        "harga_satuan"
    ]
)

# Menggunakan StringIndexer yang sudah dilatih sebelumnya
df_baru = model_indexer_menu.transform(df_baru)
df_baru = model_indexer_cabang.transform(df_baru)
df_baru = model_indexer_waktu.transform(df_baru)

# Menggabungkan fitur
df_baru = assembler.transform(df_baru)

# Membuat prediksi
hasil_prediksi = model.transform(df_baru)

hasil_prediksi.select(
    "nama_menu",
    "cabang",
    "waktu_pesan",
    "qty",
    "harga_satuan",
    "prediction"
).show(truncate=False)

+----------+----------+-----------+---+------------+------------------+
|nama_menu |cabang    |waktu_pesan|qty|harga_satuan|prediction        |
+----------+----------+-----------+---+------------+------------------+
|Kopi Susu |Magelang  |Pagi       |2  |15000       |22410.747643616138|
|Americano |Yogyakarta|Siang      |3  |20000       |60653.79162669493 |
|Cappuccino|Semarang  |Malam      |1  |25000       |31368.28664276724 |
+----------+----------+-----------+---+------------+------------------+



In [8]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
